# Sanad سند — question answering over Saudi law

## Part 6: how often should the assistant say "not found"?

The share of "not found" examples in fine-tuning is a dial. Part 3 used 25% and the model refused too often on human questions (44% false "not found" on Najiz). Part 4 used 12% and it refused too rarely (it said "not found" for only 46% of Najiz questions whose law had been removed). This notebook trains a third point on the dial, 18%, with the same plain-language rewrites as part 4, and then writes the example answers shown on the demo page.

**Input:** the output of part 4 (the rewrites and the 12% adapter). **Settings:** Internet on, GPU accelerator.

Code: https://github.com/readaloush/sanad

In [ ]:
!pip -q install sentence-transformers rank_bm25 peft bitsandbytes accelerate
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import glob, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
REWRITES = glob.glob('/kaggle/input/**/rewrites.json', recursive=True)[0]
LORA2 = glob.glob('/kaggle/input/**/lora2/adapter_config.json', recursive=True)[0].rsplit('/', 1)[0]
print(REWRITES, LORA2)

## 1. Fine-tune with 18% "not found" examples

In [ ]:
!python -m sanad.sft --model ALLaM-AI/ALLaM-7B-Instruct-preview --rewrites {REWRITES} --p-unanswerable 0.18 --data data --out results/lora3 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!python -m sanad.answer_eval --models ALLaM-AI/ALLaM-7B-Instruct-preview --adapter results/lora3 --tag _lora3 --skip-mcq --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## 2. Example answers for the demo page (before fine-tuning, 12%, 18%)

In [ ]:
!python -m sanad.demo_answers --adapters lora2={LORA2} lora3=results/lora3 --data data --out results/demo.json 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## Keep the results

In [ ]:
!mkdir -p /kaggle/working/out && cp results/answers_lora3.json results/answer_samples_lora3.json results/demo.json /kaggle/working/out/ && cp -r results/lora3 /kaggle/working/out/lora3
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out